<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ%20Uzmanl%C4%B1%C4%9F%C4%B1-5B2C1E?style=for-the-badge&logo=python&logoColor=white" alt="ECS VB&YZ 90"/>

# Hafta 6: Diyabet Teşhisi

**MAKİNE ÖĞRENMESİ UZMANLIĞI** · Modül 6 · 6 Saat

---

<a href="https://colab.research.google.com/github/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta06/hafta06_diyabet_teshisi.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Colab'da Aç"/></a>&nbsp;
<a href="https://github.com/DrMuratAltun/VB-YZ-90/blob/main/notebooks/hafta06/hafta06_diyabet_teshisi.ipynb"><img src="https://img.shields.io/badge/GitHub'da%20A%C3%A7-181717?style=flat&logo=github&logoColor=white" alt="GitHub'da Aç"/></a>&nbsp;
<a href="https://raw.githubusercontent.com/DrMuratAltun/VB-YZ-90/main/web/public/sunumlar/hafta06_siniflandirma_kaggle.pdf"><img src="https://img.shields.io/badge/PDF%20Sunum-EC1C24?style=flat&logo=adobeacrobatreader&logoColor=white" alt="PDF Sunum"/></a>&nbsp;
<a href="https://drmurataltun.github.io/VB-YZ-90/hafta/06/"><img src="https://img.shields.io/badge/Web%20Sitesi-2B7A78?style=flat&logo=googlechrome&logoColor=white" alt="Web Sitesi"/></a>

</div>

---

**Eğitmen:** Dr. Murat Altun · [yapayzekaokulum.com](https://yapayzekaokulum.com) · [GitHub](https://github.com/DrMuratAltun)

**Program:** ECS Veri Bilimi ve Yapay Zeka Uzmanlığı · 90 Saat · 15 Hafta
---

> **Bu defterde neler öğreneceksiniz?**
>
> - Pima Indians Diabetes veri seti
> - Sınıflandırma algoritmaları karşılaştırma
> - Model performans değerlendirmesi

# Hafta 6 — Diyabet Teşhisi Projesi

Bu defterde **gerçek** Pima Indians Diabetes veri setini kullanarak diyabet teşhisi yapacağız. Üç farklı sınıflandırma algoritmasını eğitip karşılaştıracağız.

## İçindekiler
1. Kütüphanelerin Yüklenmesi
2. Gerçek Veri Setini Yükleme ve Temizleme
3. Keşifsel Veri Analizi (EDA)
4. Veri Ön İşleme (StandardScaler)
5. Üç Model Eğitimi: Lojistik Regresyon, KNN, Karar Ağacı
6. Model Karşılaştırması (Sınıflandırma Raporu + Karışıklık Matrisi)
7. ROC Eğrileri — Tüm Modeller
8. En İyi Model Seçimi

## Veri Seti Hakkında
- **Kaynak:** Ulusal Diyabet, Sindirim ve Böbrek Hastalıkları Enstitüsü (NIDDK)
- **Hedef:** 21 yaş üstü Pima Kızılderili kadınlarında diyabet tahmini
- **Boyut:** 768 kayıt, 8 özellik + 1 hedef değişken (Outcome)
- **Özellikler:** Hamilelik sayısı, glukoz, kan basıncı, deri kalınlığı, insülin, BMI, soy ağacı fonksiyonu, yaş

## 1. Kütüphanelerin Yüklenmesi

Projede kullanacağımız kütüphaneleri içe aktarıyoruz:

| Kütüphane | Amacı |
|-----------|-------|
| `numpy`, `pandas` | Veri işleme ve manipülasyonu |
| `matplotlib`, `seaborn` | Görselleştirme |
| `sklearn.model_selection` | Eğitim/test ayırma |
| `sklearn.preprocessing` | Veri ölçeklendirme (StandardScaler) |
| `sklearn.linear_model` | Lojistik Regresyon modeli |
| `sklearn.neighbors` | K-En Yakın Komşu (KNN) modeli |
| `sklearn.tree` | Karar Ağacı modeli |
| `sklearn.metrics` | Başarı metrikleri (accuracy, precision, recall, ROC, AUC) |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, accuracy_score
)

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print("Kütüphaneler yüklendi!")

## 2. Gerçek Pima Indians Diabetes Verisini Yükleme

**Pima Indians Diabetes Dataset** — ABD'deki Pima Kızılderili kadınlarında diyabet teşhisi için toplanmış gerçek klinik veriler.

**Kaynak:** [Kaggle - Pima Indians Diabetes](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database)

### Özellik Açıklamaları

| Özellik | Açıklama | Birim |
|---------|----------|-------|
| Pregnancies | Hamilelik sayısı | Adet |
| Glucose | Oral glukoz tolerans testinde plazma glukoz konsantrasyonu | mg/dL |
| BloodPressure | Diyastolik kan basıncı | mmHg |
| SkinThickness | Triceps deri kıvrım kalınlığı | mm |
| Insulin | 2 saatlik serum insülini | µU/mL |
| BMI | Vücut kitle indeksi (ağırlık/boy²) | kg/m² |
| DiabetesPedigreeFunction | Diyabet soy ağacı fonksiyonu (genetik yatkınlık skoru) | — |
| Age | Yaş | Yıl |
| **Outcome** | **Hedef değişken:** 0 = Diyabet yok, 1 = Diyabet var | — |

> **Dikkat:** Bu veri setinde Glucose, BloodPressure, SkinThickness, Insulin ve BMI sütunlarındaki **0 değerleri aslında eksik veri** anlamına gelir. Bir insanın kan şekeri veya kan basıncı 0 olamaz! Bu değerleri tespit edip medyan ile dolduracağız.

In [ ]:
# Pima Indians Diabetes Dataset (gerçek veri)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 
           'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
df = pd.read_csv(url, names=columns)

print(f"Veri seti boyutu: {df.shape}")
print(f"\nHedef değişken dağılımı:")
print(df['Outcome'].value_counts().rename({0: 'Diyabet Yok', 1: 'Diyabet Var'}))

# Sıfır değerleri eksik veri olarak işaretle (biyolojik olarak imkansız değerler)
sifir_sutunlar = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
print(f"\nSıfır değer sayıları (eksik veri):")
for col in sifir_sutunlar:
    sifir_sayisi = (df[col] == 0).sum()
    print(f"  {col}: {sifir_sayisi} ({sifir_sayisi/len(df)*100:.1f}%)")

# Sıfırları NaN ile değiştir ve medyan ile doldur
for col in sifir_sutunlar:
    df[col] = df[col].replace(0, np.nan)
    df[col] = df[col].fillna(df[col].median())

print(f"\nTemizleme sonrası eksik veri: {df.isnull().sum().sum()}")
df.head()

### Temel İstatistiklere Bakalım

`df.describe()` ile her özelliğin dağılımını özetliyoruz. Dikkat etmemiz gerekenler:
- **count:** Tüm satırlarda 768 olmalı (eksik veri dolduruldu)
- **mean vs median (50%):** Fark büyükse dağılım çarpık demektir
- **min:** Temizleme sonrası 0 olmamalı (Pregnancies ve Age hariç, bunlarda 0 doğal olabilir)
- **max:** Aşırı uç değerler (outlier) var mı kontrol edelim

In [ ]:
df.describe().round(2)

## 3. Keşifsel Veri Analizi (EDA)

### 3.1 Özellik Dağılımları — Sınıfa Göre

Aşağıdaki grafikte her özelliğin dağılımını **sağlıklı (mavi)** ve **diyabetli (kırmızı)** gruplara göre ayrı ayrı çizdiriyoruz.

**Ne arıyoruz?** İki grubun belirgin şekilde ayrıştığı özellikler, modelin daha kolay öğrenebileceği özelliklerdir. Örneğin:
- **Glucose (Kan Şekeri):** Diyabetli grupta sağa kayması beklenir — yüksek glukoz diyabet belirtisi
- **BMI:** Diyabetli grupta daha yüksek olması beklenir
- **Age:** Yaşla birlikte diyabet riski artar

In [ ]:
# Özellik dağılımları — sınıfa göre ayrı renkler
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
features = df.columns[:-1]

for ax, col in zip(axes.flat, features):
    df[df['Outcome'] == 0][col].hist(ax=ax, bins=25, alpha=0.6, 
                                      color='#2196F3', label='Sağlıklı (0)')
    df[df['Outcome'] == 1][col].hist(ax=ax, bins=25, alpha=0.6, 
                                      color='#e74c3c', label='Diyabet (1)')
    ax.set_title(col)
    ax.legend(fontsize=8)

plt.suptitle('Özellik Dağılımları — Sınıfa Göre', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 3.2 Korelasyon Matrisi

Korelasyon matrisi, özellikler arasındaki doğrusal ilişkiyi **-1 ile +1** arasında ölçer:
- **+1'e yakın:** Güçlü pozitif ilişki (biri artarken diğeri de artar)
- **-1'e yakın:** Güçlü negatif ilişki (biri artarken diğeri azalır)
- **0'a yakın:** İlişki yok

**Hedef değişken (Outcome) ile korelasyona** özellikle dikkat ediyoruz. Yüksek korelasyonlu özellikler, modelin en çok yararlanacağı özelliklerdir.

> **Not:** Üst üçgeni maskeliyoruz çünkü korelasyon matrisi simetriktir — aynı bilgi iki kez gösterilmesine gerek yok.

In [ ]:
# Korelasyon matrisi — ısı haritası
plt.figure(figsize=(10, 8))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5)
plt.title('Korelasyon Matrisi')
plt.tight_layout()
plt.show()

# Hedef değişken ile korelasyonlar
print("\nOutcome ile korelasyonlar (büyükten küçüğe):")
print(corr['Outcome'].drop('Outcome').sort_values(ascending=False).round(3))

## 4. Veri Ön İşleme — StandardScaler

### Neden Ölçeklendirme Gerekli?

Özelliklerimizin **farklı ölçeklerde** olduğuna dikkat edin:
- Glucose: 44 – 199 mg/dL
- Insulin: 14 – 846 µU/mL  
- Age: 21 – 81 yıl
- BMI: 18 – 67 kg/m²

**KNN gibi mesafe tabanlı algoritmalar** bu farklılıktan çok etkilenir. Mesafe hesabında büyük ölçekli özellikler (Insulin gibi) küçük ölçeklileri (Age gibi) ezebilir.

**StandardScaler** her özelliği şu formülle dönüştürür:

$$z = \frac{x - \mu}{\sigma}$$

Bu sayede her özelliğin **ortalaması 0**, **standart sapması 1** olur.

> **Kritik Kural:** `fit_transform()` sadece **eğitim setine** uygulanır. Test setine `transform()` yapılır. Aksi halde **veri sızıntısı (data leakage)** oluşur — model test verisinden bilgi çalmış olur!

### Veri Bölme ve Ölçeklendirme Adımları

Aşağıdaki kodda sırasıyla:

1. **X ve y ayırma:** Özellikler (`X`) ve hedef değişken (`y = Outcome`) birbirinden ayrılır
2. **train_test_split:** Veri %80 eğitim, %20 test olarak ikiye bölünür. `stratify=y` parametresi, her iki sette de diyabet/sağlıklı oranının aynı kalmasını sağlar
3. **StandardScaler:** Eğitim setine `fit_transform`, test setine sadece `transform` uygulanır

In [ ]:
# Özellikler ve hedef değişkeni ayır
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Eğitim-test ayırma (%80 eğitim, %20 test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# StandardScaler — sadece eğitim setine fit, ikisine de transform
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Eğitim seti: {X_train_scaled.shape}")
print(f"Test seti: {X_test_scaled.shape}")
print(f"\nÖlçekleme sonrası ortalamalar (eğitim — 0'a yakın olmalı):")
print(np.round(X_train_scaled.mean(axis=0), 4))

## 5. Model Eğitimi — Üç Farklı Algoritma

Aynı veri üzerinde üç farklı sınıflandırma algoritmasını eğitip karşılaştıracağız:

### Algoritma Karşılaştırması

| Algoritma | Nasıl Çalışır? | Güçlü Yanı | Zayıf Yanı |
|-----------|---------------|------------|------------|
| **Lojistik Regresyon** | Doğrusal bir sınır çizerek sınıflandırır. Sigmoid fonksiyonu ile olasılık hesabı yapar. | Hızlı, yorumlanabilir, olasılık verir | Doğrusal olmayan ilişkileri yakalayamaz |
| **KNN (K-En Yakın Komşu)** | Yeni veriye en yakın k komşuya bakar, çoğunluğun sınıfını atar. | Basit, eğitim gerektirmez | Yavaş (büyük veride), k seçimi kritik |
| **Karar Ağacı** | If-else kuralları zinciri oluşturur. Her düğümde en iyi ayrıştıran özelliği seçer. | Görsel, yorumlanabilir, ölçekleme gerektirmez | Aşırı öğrenmeye (overfitting) yatkın |

Aşağıdaki kodda her model için:
- `.fit()` → Modeli eğitim verisiyle eğitir
- `.predict()` → Sınıf tahmini yapar (0 veya 1)
- `.predict_proba()[:, 1]` → Diyabet olasılığını verir (ROC eğrisi için gerekli)

In [ ]:
# Model 1: Lojistik Regresyon
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_scaled, y_train)
lr_pred = lr.predict(X_test_scaled)
lr_proba = lr.predict_proba(X_test_scaled)[:, 1]
print(f"Lojistik Regresyon Doğruluğu: {accuracy_score(y_test, lr_pred):.4f}")

# Model 2: KNN (k=7 komşu)
knn = KNeighborsClassifier(n_neighbors=7)
knn.fit(X_train_scaled, y_train)
knn_pred = knn.predict(X_test_scaled)
knn_proba = knn.predict_proba(X_test_scaled)[:, 1]
print(f"KNN (k=7) Doğruluğu:          {accuracy_score(y_test, knn_pred):.4f}")

# Model 3: Karar Ağacı (max_depth=4 ile overfitting önlenir)
dt = DecisionTreeClassifier(random_state=42, max_depth=4)
dt.fit(X_train_scaled, y_train)
dt_pred = dt.predict(X_test_scaled)
dt_proba = dt.predict_proba(X_test_scaled)[:, 1]
print(f"Karar Ağacı Doğruluğu:        {accuracy_score(y_test, dt_pred):.4f}")

## 6. Sınıflandırma Raporları ile Karşılaştırma

### Metrikleri Anlamak

| Metrik | Formül | Ne Anlama Gelir? |
|--------|--------|------------------|
| **Precision** | TP / (TP + FP) | "Diyabet var" dediğimizde ne kadar haklıyız? |
| **Recall (Duyarlılık)** | TP / (TP + FN) | Gerçekten diyabetli olanların kaçını yakaladık? |
| **F1-Score** | 2 × (P × R) / (P + R) | Precision ve Recall'un harmonik ortalaması |
| **Accuracy** | (TP + TN) / Toplam | Genel doğruluk oranı |

> **Tıbbi veri için neden Recall önemli?** Bir hastayı "sağlıklı" olarak yanlış sınıflandırmak (False Negative) çok tehlikelidir — hasta tedavi alamaz. Bu yüzden diyabet gibi sağlık projelerinde **yüksek Recall** hedefleriz.

### 6.1 Üç Modelin Sınıflandırma Raporları

In [ ]:
target_names = ['Sağlıklı (0)', 'Diyabet (1)']

all_models = {
    'Lojistik Regresyon': lr_pred,
    'KNN (k=7)': knn_pred,
    'Karar Ağacı': dt_pred
}

for name, pred in all_models.items():
    print(f"\n{'='*55}")
    print(f"{name} — Sınıflandırma Raporu")
    print('='*55)
    print(classification_report(y_test, pred, target_names=target_names))

### 6.2 Karışıklık Matrisleri (Confusion Matrix)

Karışıklık matrisi, modelin tahminlerini 4 kategoride gösterir:

|  | Tahmin: Sağlıklı | Tahmin: Diyabet |
|--|-------------------|------------------|
| **Gerçek: Sağlıklı** | Doğru Negatif (TN) | Yanlış Pozitif (FP) |
| **Gerçek: Diyabet** | Yanlış Negatif (FN) | Doğru Pozitif (TP) |

- **Sol üst (TN):** Sağlıklı hastayı doğru bildi
- **Sağ alt (TP):** Diyabetli hastayı doğru bildi
- **Sağ üst (FP):** Sağlıklı hastaya yanlışlıkla diyabet dedi (gereksiz endişe)
- **Sol alt (FN):** Diyabetli hastayı kaçırdı **(en tehlikeli hata!)**

In [ ]:
# Karışıklık matrisleri — 3 model yan yana
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, pred) in zip(axes, all_models.items()):
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd', ax=ax,
                xticklabels=target_names, yticklabels=target_names)
    ax.set_title(f'{name}')
    ax.set_xlabel('Tahmin')
    ax.set_ylabel('Gerçek')

plt.suptitle('Karışıklık Matrisleri', fontsize=14)
plt.tight_layout()
plt.show()

## 7. ROC Eğrileri — Tüm Modeller Aynı Grafikte

### ROC Eğrisi ve AUC Nedir?

**ROC (Receiver Operating Characteristic) eğrisi**, modelin farklı eşik değerlerinde performansını gösterir:
- **X ekseni:** Yanlış Pozitif Oranı (FPR) — sağlıklı hastalar arasında yanlışlıkla diyabet denenler
- **Y ekseni:** Doğru Pozitif Oranı (TPR) — diyabetli hastalar arasında doğru yakalananlar

**AUC (Area Under Curve)**, eğrinin altındaki alan:
- **AUC = 1.0:** Mükemmel model (tüm hastaları doğru sınıflandırır)
- **AUC = 0.5:** Rastgele tahmin (yazı-tura kadar başarılı)
- **AUC > 0.8:** İyi bir model
- **AUC > 0.9:** Çok iyi bir model

> Eğri sol üst köşeye ne kadar yakınsa model o kadar başarılıdır.

In [ ]:
# ROC eğrileri — 3 model aynı grafikte
plt.figure(figsize=(10, 8))

all_probas = {
    'Lojistik Regresyon': lr_proba,
    'KNN (k=7)': knn_proba,
    'Karar Ağacı': dt_proba
}

colors = ['#2196F3', '#4CAF50', '#FF9800']

for (name, proba), color in zip(all_probas.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, proba)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, linewidth=2.5,
             label=f'{name} (AUC = {roc_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.7, label='Rastgele (AUC = 0.500)')

plt.xlabel('Yanlış Pozitif Oranı (FPR)', fontsize=13)
plt.ylabel('Doğru Pozitif Oranı (TPR)', fontsize=13)
plt.title('ROC Eğrileri — Diyabet Teşhisi Modelleri', fontsize=14)
plt.legend(loc='lower right', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. En İyi Model Seçimi

### Karar Nasıl Verilir?

Sadece **Accuracy (doğruluk)** yeterli değildir. Dengesiz veri setlerinde (burada ~%65 sağlıklı, ~%35 diyabet) yüksek doğruluk yanıltıcı olabilir — model herkese "sağlıklı" dese bile %65 doğruluk elde eder!

**AUC skoru** daha güvenilir bir metriktir çünkü eşik değerinden bağımsızdır ve sınıf dengesizliğinden daha az etkilenir.

### 8.1 Sonuç Tablosu

In [ ]:
# Tüm metrikleri bir tabloda topla
results = []

for (name, pred), (_, proba) in zip(all_models.items(), all_probas.items()):
    fpr, tpr, _ = roc_curve(y_test, proba)
    roc_auc = auc(fpr, tpr)
    acc = accuracy_score(y_test, pred)
    results.append({
        'Model': name,
        'Doğruluk': acc,
        'AUC': roc_auc
    })

results_df = pd.DataFrame(results).sort_values('AUC', ascending=False)
print("Model Karşılaştırma Tablosu")
print("=" * 50)
print(results_df.to_string(index=False))

best_model = results_df.iloc[0]['Model']
best_auc = results_df.iloc[0]['AUC']
print(f"\nEn İyi Model: {best_model} (AUC = {best_auc:.3f})")

### 8.2 Görsel Karşılaştırma

Üç modelin doğruluk değerlerini çubuk grafikle karşılaştırıyoruz. Çubukların üstündeki sayılar kesin doğruluk değerleridir.

In [ ]:
# Doğruluk karşılaştırma çubuk grafiği
plt.figure(figsize=(10, 5))
colors = ['#2196F3', '#4CAF50', '#FF9800']
bars = plt.bar(results_df['Model'], results_df['Doğruluk'], color=colors, edgecolor='white', width=0.5)

for bar, val in zip(bars, results_df['Doğruluk']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005, 
             f'{val:.3f}', ha='center', fontsize=13, fontweight='bold')

plt.ylabel('Doğruluk (Accuracy)')
plt.title('Model Doğruluk Karşılaştırması — Diyabet Teşhisi')
plt.ylim(0, 1.05)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Özet ve Önemli Çıkarımlar

Bu defterde gerçek Pima Indians Diabetes veri setiyle uçtan uca bir sınıflandırma projesi gerçekleştirdik:

### Adımlar
1. **Gerçek veri yükleme:** Kaggle'dan Pima Indians Diabetes veri seti (768 kayıt)
2. **Veri temizleme:** Biyolojik olarak imkansız 0 değerlerini tespit edip medyanla doldurduk
3. **EDA:** Özellik dağılımları ve korelasyon analizi ile veriyi anladık
4. **Ön işleme:** StandardScaler ile özellikleri normalize ettik
5. **3 model eğitimi:** Lojistik Regresyon, KNN ve Karar Ağacı
6. **Değerlendirme:** Sınıflandırma raporu, karışıklık matrisi, ROC eğrisi ve AUC

### Önemli Dersler

| Ders | Açıklama |
|------|----------|
| **Veri temizliği kritiktir** | Gerçek veride 0 değerleri eksik veri olarak gizlenebilir |
| **Ölçekleme şarttır** | KNN gibi mesafe tabanlı algoritmalar ölçeklemeye ihtiyaç duyar |
| **Tek metriğe güvenmeyin** | Accuracy yanıltıcı olabilir; AUC, Recall ve F1'e de bakın |
| **Tıbbi veride Recall önemli** | Diyabetli bir hastayı kaçırmak, sağlıklı birine "diyabet" demekten daha tehlikelidir |
| **Basit modeller yeterli olabilir** | Lojistik Regresyon gibi basit modeller çoğu zaman karmaşık modellerle rekabet eder |

### Alıştırmalar
1. K değerini değiştirerek (k=3, 5, 9, 11) KNN'in performansını karşılaştırın
2. Karar Ağacı'nda `max_depth` parametresini değiştirip aşırı öğrenme etkisini gözlemleyin
3. **Bonus:** `RandomForestClassifier` ekleyip 4. model olarak karşılaştırın

---

<div align="center">

<img src="https://img.shields.io/badge/ECS-Veri%20Bilimi%20%26%20YZ-5B2C1E?style=flat-square&logo=python&logoColor=white" alt="ECS"/>

**Dr. Murat Altun** · Veri Bilimi ve Yapay Zeka Eğitmeni

<a href="https://yapayzekaokulum.com">Yapay Zeka Okulum</a> ·
<a href="https://gencyz.com">GençYZ</a> ·
<a href="https://yz-araclari.com">YZ Araçları</a> ·
<a href="https://scholargent.com">ScholarAI</a> ·
<a href="https://drmurataltun.github.io">Kişisel Site</a>

<a href="https://drmurataltun.github.io/VB-YZ-90/">drmurataltun.github.io/VB-YZ-90</a>

---

*Bu materyal ECS Veri Bilimi ve Yapay Zeka Uzmanlığı Programı için hazırlanmıştır.*

&copy; 2026 Dr. Murat Altun. Tüm hakları saklıdır.

</div>